# Demonstração de Vazamento de Dados (Data Leakage)\n
Neste notebook, vamos replicar a falha metodológica encontrada nos notebooks de referência. Vamos concatenar os conjuntos de **Treino (Motores 1-10)** e **Teste (Motores 11-20)** antes da divisão, permitindo que a Rede Neural 'cole na prova' e atinja um R² ilusório.

In [1]:
import os
import h5py
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

I0000 00:00:1789511916.544450   31380 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789511916.709258   31380 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 1. O 'Pecado Capital': Concatenando Treino e Teste

In [2]:
# Carregar arquivo
filename = 'data/N-CMAPSS_DS02-006.h5'
with h5py.File(filename, 'r') as hdf:
    W_dev = np.array(hdf.get('W_dev'))
    X_s_dev = np.array(hdf.get('X_s_dev'))
    Y_dev = np.array(hdf.get('Y_dev'))
    A_dev = np.array(hdf.get('A_dev'))
    
    W_test = np.array(hdf.get('W_test'))
    X_s_test = np.array(hdf.get('X_s_test'))
    Y_test = np.array(hdf.get('Y_test'))
    A_test = np.array(hdf.get('A_test'))

print("ATENÇÃO: Concatenando DEV e TEST numa matriz única!")
W_full = np.concatenate((W_dev, W_test), axis=0)  
X_s_full = np.concatenate((X_s_dev, X_s_test), axis=0)
Y_full = np.concatenate((Y_dev, Y_test), axis=0) 
A_full = np.concatenate((A_dev, A_test), axis=0)

del W_dev, W_test, X_s_dev, X_s_test, Y_dev, Y_test, A_dev, A_test
gc.collect()

ATENÇÃO: Concatenando DEV e TEST numa matriz única!


0

## 2. Geração de Features (Base Viciada)

In [3]:
def create_temporal_features_safe(W, X_s, Y, A, window=40):
    A_features = A[:, 1:].astype('float32') # Traz o ciclo, classe de voo e hs
    matriz_base = np.concatenate((W, X_s, A_features), axis=1).astype('float32')
    
    df = pd.DataFrame(matriz_base)
    df['unit'] = A[:, 0].astype('float32')
    df['RUL'] = Y.flatten().astype('float32')
    
    print("Calculando médias e variâncias (Modo Economia de RAM)...")
    df_mean = df.groupby('unit').rolling(window=window, min_periods=1).mean().reset_index(level=0, drop=True).sort_index().astype('float32')
    df_var = df.groupby('unit').rolling(window=window, min_periods=1).var().fillna(0).reset_index(level=0, drop=True).sort_index().astype('float32')
    
    df_mean = df_mean[df.columns[:-2]]
    df_var = df_var[df.columns[:-2]]
    df_raw = pd.DataFrame(matriz_base)
    
    X_temporal = pd.concat([df_raw, df_mean, df_var], axis=1).values.astype('float32')
    y_labels = df['RUL'].values.astype('float32')
    
    del df, df_mean, df_var, df_raw, matriz_base
    gc.collect()
    
    return X_temporal, y_labels

print("Processando Matriz Viciada Completa...")
X_full_raw, y_full = create_temporal_features_safe(W_full, X_s_full, Y_full, A_full, window=20)

del W_full, X_s_full, Y_full, A_full
gc.collect()

Processando Matriz Viciada Completa...
Calculando médias e variâncias (Modo Economia de RAM)...


0

## 3. Escalonamento e O Falso 'Train/Test Split'\n
Aqui o modelo vai receber no Teste, recortes dos mesmos voos que ele já decorou no Treino.

In [4]:
# Escalonamento Inicial
scaler = StandardScaler()
X_full_scaled = scaler.fit_transform(X_full_raw).astype('float32')

del X_full_raw
gc.collect()

print("Misturando os 20 motores num caldeirão só (80% treino, 20% teste)...")
X_train, X_test, y_train, y_test = train_test_split(
    X_full_scaled, y_full, test_size=0.2, random_state=42
)

del X_full_scaled, y_full
gc.collect()

Misturando os 20 motores num caldeirão só (80% treino, 20% teste)...


0

## 4. Treinamento da MLP

In [5]:
final_model = Sequential()
final_model.add(tf.keras.layers.Input(shape=(X_train.shape[1],)))

final_model.add(Dense(units=512, activation='relu'))
final_model.add(BatchNormalization())
final_model.add(Dropout(0.2))

final_model.add(Dense(units=256, activation='relu'))
final_model.add(BatchNormalization())
final_model.add(Dropout(0.2))

final_model.add(Dense(units=128, activation='relu'))
final_model.add(BatchNormalization())

final_model.add(Dense(1, activation='linear'))

final_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])

stop_early = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

print("Iniciando Treinamento Viciado...")
history = final_model.fit(
    X_train, y_train,
    validation_split=0.2, 
    epochs=100, 
    batch_size=4096, 
    callbacks=[stop_early],
    verbose=1
)

I0000 00:00:1789511938.736585   31380 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5743 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a


Iniciando Treinamento Viciado...
Epoch 1/100


I0000 00:00:1789511940.942768   31540 service.cc:153] XLA service 0x5c8f2cc46280 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789511940.942825   31540 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5050 Laptop GPU, Compute Capability 12.0a (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.9.0; DNN: 9.17.0)
I0000 00:00:1789511941.075451   31540 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1789511941.446754   31540 cuda_dnn.cc:461] Loaded cuDNN version 91700
I0000 00:00:1789511941.511974   31540 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_3060__.30
I0000 00:00:1789511944.417399   31644 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_30', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1789511944.516741   31638 subprocess_compilation.cc:348] 

  44/1019 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1538.8676 - mae: 36.0127  

I0000 00:00:1789511948.683453   31540 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1010/1019 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 751.5968 - mae: 23.0628

I0000 00:00:1789511953.245768   31540 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_3060__.30
I0000 00:00:1789511953.466260   31944 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_28', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1789511953.550033   31942 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_28', 24 bytes spill stores, 24 bytes spill loads

I0000 00:00:1789511953.581089   31935 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_28', 76 bytes spill stores, 76 bytes spill loads

I0000 00:00:1789511953.849784   31941 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_20', 16 bytes spill stores, 16 bytes spill loads

I0000 00:00:1789511954.521129   31945 subprocess_compilat

1019/1019 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 747.8442 - mae: 22.9698

I0000 00:00:1789511961.223148   32286 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_12', 16 bytes spill stores, 16 bytes spill loads

I0000 00:00:1789511962.039846   32272 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_10', 16 bytes spill stores, 16 bytes spill loads

I0000 00:00:1789511962.079132   32286 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_10', 28 bytes spill stores, 20 bytes spill loads

I0000 00:00:1789511962.131266   32280 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_10', 160 bytes spill stores, 144 bytes spill loads



1019/1019 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 325.7408 - mae: 12.5089 - val_loss: 31.7756 - val_mae: 3.6156
Epoch 2/100
1019/1019 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 28.6668 - mae: 3.6409 - val_loss: 26.9102 - val_mae: 3.3908
Epoch 3/100
1019/1019 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 24.0465 - mae: 3.2729 - val_loss: 23.8305 - val_mae: 3.2193
Epoch 4/100
1019/1019 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 21.1890 - mae: 3.0372 - val_loss: 16.6666 - val_mae: 2.6611
Epoch 5/100
1019/1019 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 19.1551 - mae: 2.8642 - val_loss: 13.8049 - val_mae: 2.2397
Epoch 6/100
1019/1019 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 17.5361 - mae: 2.7249 - val_loss: 13.9835 - val_mae: 2.3016
Epoch 7/100
1019/1019 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 16.3028 - mae: 2.6141 - val_loss: 11.9677 - val_mae: 2.0482
Epoch 8/100
1019/1019 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 15.2164 - mae: 2.5131 - val_loss: 10.7832 - val_mae: 1.9638
Epoch 9/100
1019/1019 ━━

## 5. O Resultado Ilusório\n
Como a Rede Neural já 'viu' partes dos voos do conjunto de teste durante o treinamento, a métrica final será artificialmente alta.

In [6]:
y_pred = final_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"❌ MAE Viciado: {mae:.2f}")
print(f"❌ RMSE Viciado: {rmse:.2f}")
print(f"❌ R² Score Falso (Data Leakage): {r2:.4f}")

40733/40733 ━━━━━━━━━━━━━━━━━━━━ 51s 1ms/step
❌ MAE Viciado: 1.40
❌ RMSE Viciado: 2.46
❌ R² Score Falso (Data Leakage): 0.9874
